In [ ]:
!nvidia-smi

Sun Jun  7 16:55:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             53W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%%capture
import os
if not os.path.exists('/content/drive/MyDrive/AAAI_TALAS/AAAI-TALAS-main'):
    !unzip /content/drive/MyDrive/AAAI_TALAS/AAAI-TALAS-main.zip

In [ ]:
%cd /content/AAAI-TALAS-main

/content/AAAI-TALAS-main


In [ ]:
# Remove stale cached teacher embeddings before training
!rm -rf cache
!mkdir -p cache
!echo "Removed cache/"

Removed cache/


In [ ]:
!pip install -r requirements.txt

In [ ]:
!sed -i 's|\.\./main.py|main.py|g' scripts/train_talas.sh
!sed -i 's|TRAIN_DATA=.*|TRAIN_DATA="data/merged_9_data_3k_each_ver2.csv"|g' scripts/train_talas.sh
!sed -i 's|TEACHER_MODEL=.*|TEACHER_MODEL="Qwen/Qwen3-Embedding-4B"|g' scripts/train_talas.sh
!sed -i 's|STUDENT_MODEL=.*|STUDENT_MODEL="google-bert/bert-base-uncased"|g' scripts/train_talas.sh
!cat scripts/train_talas.sh | grep -E "TRAIN_DATA|TEACHER_MODEL|STUDENT_MODEL"

!sed -i -E 's/BATCH_SIZE=[0-9]+/BATCH_SIZE=256/g' scripts/train_talas.sh
!sed -i -E 's/(--batch_size\s+)[0-9]+/\1256/g' scripts/train_talas.sh
!sed -i -E 's/(--per_device_train_batch_size\s+)[0-9]+/\1256/g' scripts/train_talas.sh

# Kiểm tra lại các dòng có chứa từ khoá batch trong script
!cat scripts/train_talas.sh | grep -i "batch"
!test -f data/test_debug.csv || (echo 'Missing data/test_debug.csv. Re-run the unzip cell or check the archive.' && false)
!mkdir -p analysis/talas_diagnostics
!bash -c 'set -o pipefail; bash scripts/train_talas.sh 2>&1 | tee analysis/talas_diagnostics/train_talas.log'

TRAIN_DATA="data/merged_9_data_3k_each_ver2.csv"
STUDENT_MODEL="google-bert/bert-base-uncased"
TEACHER_MODEL="Qwen/Qwen3-Embedding-4B"
    --train_data $TRAIN_DATA \
    --student_model $STUDENT_MODEL \
    --teacher_model $TEACHER_MODEL \
BATCH_SIZE=256
    --batch_size $BATCH_SIZE \
Training with TALAS method

Configuration for TALAS method:
  train_data_path           : data/merged_9_data_3k_each_ver2.csv
  student_model_name        : google-bert/bert-base-uncased
  teacher_model_name        : Qwen/Qwen3-Embedding-4B
  batch_size                : 256
  epochs                    : 5
  learning_rate             : 2e-05
  max_length                : 256
  save_dir                  : checkpoints/talas

Done setup_seed with seed=42
[WARN] Only 1 GPU available -> both on cuda:0
Done setup_devices
Loading tokenizers...
Loading student model: google-bert/bert-base-uncased
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6179.97it/s]
[transformers] BertModel LOAD REPORT from: google-b

In [ ]:
# TALAS / LLASD diagnostics: log + visualize post-training signals.
# This notebook treats LLASD as TeacherAnchorKD.loss_struct: MSE between
# pairwise in-batch relation matrices of consecutive student CLS layers.

import gc
import json
import math
import re
from contextlib import nullcontext
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy.stats import pearsonr, spearmanr
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

from src.evaluation.evaluation_automodel import eval_sts_tasks, test_sts_tasks
from src.pooling import last_token_pool, mean_pooling

DIAG_DIR = Path("analysis/talas_diagnostics")
DIAG_DIR.mkdir(parents=True, exist_ok=True)

# Keep these modest for fast Colab probes; increase after the pipeline works.
TRAIN_SAMPLE_TEXTS = 512
STS_SAMPLE_PER_FILE = 512
DIAG_BATCH_SIZE = 64
GRAD_BATCH_SIZE = 32

# B. Geometry smoothing vs semantic hierarchy.
GEOMETRY_SCORE_EPS = 1e-12

# C. Stable vs unstable relation analysis.
RUN_STABILITY_DIAGNOSTICS = True
RUN_DOMAIN_STABILITY_ON_STS = True
RELATION_STABILITY_SAMPLE_TEXTS = 256
RELATION_REPEATS = 5
STS_STABILITY_SAMPLE_PER_FILE = 192
STS_STABILITY_REPEATS = 3
STABLE_RELATION_QUANTILE = 0.20
UNSTABLE_RELATION_QUANTILE = 0.80
AUGMENT_DROP_PROB = 0.12

RUN_TEACHER_STS_CALIBRATION = True
RANDOM_SEED = 42

checkpoint_dir = Path("checkpoints/talas")
best_checkpoint = checkpoint_dir / "best_model.pt"
if best_checkpoint.exists():
    checkpoint_path = best_checkpoint
else:
    checkpoints = sorted(
        checkpoint_dir.glob("checkpoint_epoch_*.pt"),
        key=lambda p: int(re.search(r"checkpoint_epoch_(\d+)", p.stem).group(1)),
    )
    assert checkpoints, f"No checkpoint found in {checkpoint_dir}"
    checkpoint_path = checkpoints[-1]

try:
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
except TypeError:
    checkpoint = torch.load(checkpoint_path, map_location="cpu")

cfg = checkpoint.get("config", {})
student_model_name = cfg.get("student_model_name", "google-bert/bert-base-uncased")
teacher_model_name = cfg.get("teacher_model_name", "Qwen/Qwen3-Embedding-0.6B")
train_data_path = Path(cfg.get("train_data_path", "data/merged_3_data_5k_each.csv"))
task_type = cfg.get("task_type", "pair_cls")
max_length = int(cfg.get("max_length", 256))
start_rkd = int(cfg.get("start_rkd", 0))
teacher_dtype_name = cfg.get("teacher_dtype", "bfloat16")
teacher_pooling = cfg.get("pooling_method", "last_token")

print(f"Diagnostics checkpoint: {checkpoint_path}")
print(f"Student: {student_model_name}")
print(f"Teacher for STS calibration: {teacher_model_name}")
print(f"Training data sample source: {train_data_path}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
student_tokenizer = AutoTokenizer.from_pretrained(student_model_name)
student_model = AutoModel.from_pretrained(student_model_name)
student_model.load_state_dict(checkpoint["model_state_dict"])
student_model.to(device)
student_model.eval()


def autocast_for(device):
    if device.type == "cuda":
        return torch.amp.autocast("cuda", dtype=torch.float16)
    return nullcontext()


def safe_corr(x, y):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0:
        return {"pearson": np.nan, "spearman": np.nan, "n": int(len(x))}
    return {
        "pearson": float(pearsonr(x, y).statistic),
        "spearman": float(spearmanr(x, y).statistic),
        "n": int(len(x)),
    }


def get_first_side_texts(df, task_type):
    if task_type == "single_cls":
        return df["text"].astype(str).tolist()
    if "premise" in df.columns:
        return df["premise"].astype(str).tolist()
    if "sentence1" in df.columns:
        return df["sentence1"].astype(str).tolist()
    if "text" in df.columns:
        return df["text"].astype(str).tolist()
    return df.iloc[:, 0].astype(str).tolist()


def batched(iterable, batch_size):
    for start in range(0, len(iterable), batch_size):
        yield iterable[start:start + batch_size]


def encode_texts(tokenizer, texts, max_length, device):
    encoded = tokenizer(
        list(texts),
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors="pt",
    )
    return {k: v.to(device) for k, v in encoded.items() if torch.is_tensor(v)}


def collect_student_cls_layers(model, tokenizer, texts, batch_size, max_length, device):
    layer_chunks = None
    model.eval()
    with torch.no_grad():
        for batch_texts in tqdm(list(batched(texts, batch_size)), desc="Student hidden-state probe"):
            batch = encode_texts(tokenizer, batch_texts, max_length, device)
            with autocast_for(device):
                out = model(**batch, output_hidden_states=True, return_dict=True)
            cls_layers = [h[:, 0, :].detach().float().cpu() for h in out.hidden_states]
            if layer_chunks is None:
                layer_chunks = [[] for _ in cls_layers]
            for idx, cls in enumerate(cls_layers):
                layer_chunks[idx].append(cls)
    return [torch.cat(chunks, dim=0) for chunks in layer_chunks]


def linear_cka(x, y, eps=1e-12):
    x = x.float() - x.float().mean(dim=0, keepdim=True)
    y = y.float() - y.float().mean(dim=0, keepdim=True)
    xty = x.T @ y
    xtx = x.T @ x
    yty = y.T @ y
    numerator = torch.sum(xty * xty)
    denominator = torch.linalg.matrix_norm(xtx) * torch.linalg.matrix_norm(yty)
    return float((numerator / denominator.clamp_min(eps)).cpu())


def relation_matrix(x, eps=1e-12):
    x = F.normalize(x.float(), dim=-1, eps=eps)
    return x @ x.T


def upper_triangle_values(matrix):
    n = matrix.shape[0]
    idx = torch.triu_indices(n, n, offset=1, device=matrix.device)
    return matrix[idx[0], idx[1]]


def pair_group_contribution(cls_layers, start_rkd=0):
    rows = []
    for layer_idx in range(start_rkd, len(cls_layers) - 1):
        source_relation = relation_matrix(cls_layers[layer_idx])
        target_relation = relation_matrix(cls_layers[layer_idx + 1])
        source_vals = upper_triangle_values(source_relation)
        target_vals = upper_triangle_values(target_relation)
        sq_error = (source_vals - target_vals).pow(2)
        if sq_error.numel() == 0:
            continue
        q1, q2 = torch.quantile(target_vals.float(), torch.tensor([1 / 3, 2 / 3], device=target_vals.device))
        groups = {
            "bottom": target_vals <= q1,
            "middle": (target_vals > q1) & (target_vals <= q2),
            "top": target_vals > q2,
        }
        total_sse = float(sq_error.sum().item())
        for group_name, mask in groups.items():
            count = int(mask.sum().item())
            group_sse = float(sq_error[mask].sum().item()) if count else 0.0
            rows.append({
                "layer_from": layer_idx,
                "layer_to": layer_idx + 1,
                "layer_pair": f"{layer_idx}->{layer_idx + 1}",
                "group": group_name,
                "count": count,
                "mean_target_relation": float(target_vals[mask].mean().item()) if count else np.nan,
                "mean_sq_error": float(sq_error[mask].mean().item()) if count else np.nan,
                "sse": group_sse,
                "frac_sse": group_sse / total_sse if total_sse else np.nan,
            })
    return pd.DataFrame(rows)


def llasd_grad_norm_by_layer(model, tokenizer, texts, batch_size, max_length, device, start_rkd=0):
    model.zero_grad(set_to_none=True)
    model.eval()
    batch = encode_texts(tokenizer, texts[:batch_size], max_length, device)
    out = model(**batch, output_hidden_states=True, return_dict=True)
    cls_layers = [h[:, 0, :] for h in out.hidden_states]
    loss_terms = []
    for layer_idx in range(start_rkd, len(cls_layers) - 1):
        source_relation = relation_matrix(cls_layers[layer_idx])
        target_relation = relation_matrix(cls_layers[layer_idx + 1])
        loss_terms.append(F.mse_loss(source_relation, target_relation))
    loss_struct = torch.stack(loss_terms).mean()
    loss_struct.backward()

    rows_by_layer = {}
    for name, param in model.named_parameters():
        if param.grad is None:
            continue
        if name.startswith("embeddings."):
            layer_key = "embeddings"
            layer_order = -1
        else:
            match = re.search(r"encoder\.layer\.(\d+)\.", name)
            if match:
                layer_order = int(match.group(1))
                layer_key = f"layer_{layer_order}"
            else:
                layer_key = "other"
                layer_order = 10_000
        row = rows_by_layer.setdefault(layer_key, {
            "layer": layer_key,
            "layer_order": layer_order,
            "grad_sq_sum": 0.0,
            "param_sq_sum": 0.0,
            "n_params_with_grad": 0,
        })
        row["grad_sq_sum"] += float(param.grad.detach().float().pow(2).sum().cpu())
        row["param_sq_sum"] += float(param.detach().float().pow(2).sum().cpu())
        row["n_params_with_grad"] += int(param.numel())

    rows = []
    for row in rows_by_layer.values():
        grad_norm = math.sqrt(row.pop("grad_sq_sum"))
        param_norm = math.sqrt(row.pop("param_sq_sum"))
        row["grad_norm"] = grad_norm
        row["param_norm"] = param_norm
        row["grad_param_ratio"] = grad_norm / max(param_norm, 1e-12)
        row["loss_struct_probe"] = float(loss_struct.detach().cpu())
        rows.append(row)
    model.zero_grad(set_to_none=True)
    return pd.DataFrame(rows).sort_values("layer_order").reset_index(drop=True)


def collect_embeddings(
    model,
    tokenizer,
    texts,
    batch_size,
    max_length,
    device,
    pooling="cls",
    training_mode=False,
    desc=None,
):
    chunks = []
    was_training = model.training
    model.train(training_mode)
    try:
        with torch.no_grad():
            iterator = list(batched(texts, batch_size))
            probe_name = desc or f"Embedding probe ({pooling}, {'train' if training_mode else 'eval'})"
            for batch_texts in tqdm(iterator, desc=probe_name):
                batch = encode_texts(tokenizer, batch_texts, max_length, device)
                with autocast_for(device):
                    out = model(**batch, return_dict=True)
                if pooling == "last_token":
                    emb = last_token_pool(out.last_hidden_state, batch["attention_mask"])
                elif pooling == "mean":
                    emb = mean_pooling(out.last_hidden_state, batch["attention_mask"])
                else:
                    emb = out.last_hidden_state[:, 0, :]
                chunks.append(F.normalize(emb.float(), dim=-1).detach().cpu())
    finally:
        model.train(was_training)
    return torch.cat(chunks, dim=0)


def relation_vector_from_embeddings(embeddings):
    return upper_triangle_values(relation_matrix(embeddings)).detach().float().cpu()


def minmax_01(values, eps=GEOMETRY_SCORE_EPS):
    values = np.asarray(values, dtype=np.float64)
    finite = np.isfinite(values)
    if not finite.any():
        return np.full_like(values, np.nan, dtype=np.float64)
    lo = np.nanmin(values[finite])
    hi = np.nanmax(values[finite])
    if hi - lo < eps:
        return np.zeros_like(values, dtype=np.float64)
    return (values - lo) / (hi - lo)


def relation_geometry_diagnostics(cls_layers, cka, diag_dir):
    relation_mats = [relation_matrix(layer).detach().float().cpu() for layer in cls_layers]
    num_layers = len(relation_mats)
    n_items = relation_mats[0].shape[0]

    frob = np.zeros((num_layers, num_layers), dtype=np.float64)
    frob_rms = np.zeros((num_layers, num_layers), dtype=np.float64)
    for i in range(num_layers):
        for j in range(i, num_layers):
            value = float(torch.linalg.matrix_norm(relation_mats[i] - relation_mats[j], ord="fro").item())
            frob[i, j] = frob[j, i] = value
            frob_rms[i, j] = frob_rms[j, i] = value / math.sqrt(n_items * n_items)

    labels = [f"L{i}" for i in range(num_layers)]
    frob_df = pd.DataFrame(frob, index=labels, columns=labels)
    frob_rms_df = pd.DataFrame(frob_rms, index=labels, columns=labels)
    frob_df.to_csv(diag_dir / "relation_geometry_distance_frobenius.csv")
    frob_rms_df.to_csv(diag_dir / "relation_geometry_distance_rms.csv")

    plt.figure(figsize=(9, 7))
    plt.imshow(frob_rms_df.values, cmap="viridis")
    plt.colorbar(label=r"$||R_i - R_j||_F / N$")
    plt.xticks(range(num_layers), labels, rotation=90)
    plt.yticks(range(num_layers), labels)
    plt.title(r"Layer-to-layer relation geometry distance: $D_{ij}=||R_i-R_j||_F$")
    plt.tight_layout()
    plt.savefig(diag_dir / "relation_geometry_distance_heatmap.png", dpi=180)
    plt.show()

    relation_vectors = [upper_triangle_values(mat).detach().float().cpu() for mat in relation_mats]
    relation_std = np.array([float(vec.std(unbiased=False).item()) for vec in relation_vectors])
    relation_mean = np.array([float(vec.mean().item()) for vec in relation_vectors])
    relation_abs_mean = np.array([float(vec.abs().mean().item()) for vec in relation_vectors])

    prev_drift = np.full(num_layers, np.nan, dtype=np.float64)
    next_drift = np.full(num_layers, np.nan, dtype=np.float64)
    for layer_idx in range(num_layers):
        if layer_idx > 0:
            prev_drift[layer_idx] = frob_rms[layer_idx - 1, layer_idx]
        if layer_idx < num_layers - 1:
            next_drift[layer_idx] = frob_rms[layer_idx, layer_idx + 1]

    neighbor_drift = np.nanmean(np.vstack([prev_drift, next_drift]), axis=0)
    neighbor_cka = []
    for layer_idx in range(num_layers):
        vals = []
        if layer_idx > 0:
            vals.append(cka[layer_idx - 1, layer_idx])
        if layer_idx < num_layers - 1:
            vals.append(cka[layer_idx, layer_idx + 1])
        neighbor_cka.append(float(np.mean(vals)) if vals else np.nan)
    neighbor_cka = np.array(neighbor_cka, dtype=np.float64)

    drift_smoothness = 1.0 - minmax_01(neighbor_drift)
    contrast_retention_vs_input = relation_std / max(float(relation_std[0]), GEOMETRY_SCORE_EPS)
    contrast_loss_vs_input = np.maximum(0.0, 1.0 - contrast_retention_vs_input)
    over_smoothing_risk = drift_smoothness * contrast_loss_vs_input

    layer_score_df = pd.DataFrame({
        "layer": list(range(num_layers)),
        "relation_mean": relation_mean,
        "relation_abs_mean": relation_abs_mean,
        "relation_std": relation_std,
        "prev_relation_drift_rms": prev_drift,
        "next_relation_drift_rms": next_drift,
        "neighbor_relation_drift_rms": neighbor_drift,
        "neighbor_cka": neighbor_cka,
        "drift_smoothness_score": drift_smoothness,
        "contrast_retention_vs_input": contrast_retention_vs_input,
        "contrast_loss_vs_input": contrast_loss_vs_input,
        "over_smoothing_risk_score": over_smoothing_risk,
    })
    layer_score_df.to_csv(diag_dir / "layer_geometry_scores.csv", index=False)

    fig, ax1 = plt.subplots(figsize=(11, 4))
    ax1.plot(layer_score_df["layer"], layer_score_df["relation_std"], marker="o", label="relation std / geometric contrast", color="#2A6F97")
    ax1.plot(layer_score_df["layer"], layer_score_df["neighbor_relation_drift_rms"], marker="o", label="neighbor drift RMS", color="#D1495B")
    ax1.set_xlabel("student layer")
    ax1.set_ylabel("relation geometry statistic")
    ax2 = ax1.twinx()
    ax2.plot(layer_score_df["layer"], layer_score_df["over_smoothing_risk_score"], marker="s", color="#EDAE49", label="over-smoothing risk score")
    ax2.set_ylabel("risk score")
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="best")
    ax1.set_title("Layer geometry scores: smoothing pressure vs relation contrast")
    plt.tight_layout()
    plt.savefig(diag_dir / "layer_geometry_scores.png", dpi=180)
    plt.show()

    trajectory_rows = []
    vec_stack = torch.stack(relation_vectors).float()
    for layer_idx in range(1, num_layers - 1):
        step_prev = vec_stack[layer_idx] - vec_stack[layer_idx - 1]
        step_next = vec_stack[layer_idx + 1] - vec_stack[layer_idx]
        prev_norm = float(torch.linalg.vector_norm(step_prev).item())
        next_norm = float(torch.linalg.vector_norm(step_next).item())
        denom = max(prev_norm * next_norm, GEOMETRY_SCORE_EPS)
        cosine = float(torch.dot(step_prev, step_next).item() / denom)
        cosine = max(-1.0, min(1.0, cosine))
        turn_angle = math.degrees(math.acos(cosine))
        curvature = float(torch.linalg.vector_norm(step_next - step_prev).item() / max(prev_norm + next_norm, GEOMETRY_SCORE_EPS))
        trajectory_rows.append({
            "center_layer": layer_idx,
            "prev_step_norm": prev_norm,
            "next_step_norm": next_norm,
            "turn_angle_degrees": turn_angle,
            "trajectory_curvature": curvature,
        })

    trajectory_df = pd.DataFrame(trajectory_rows)
    trajectory_df.to_csv(diag_dir / "relation_trajectory_curvature.csv", index=False)
    if not trajectory_df.empty:
        fig, ax1 = plt.subplots(figsize=(10, 4))
        ax1.plot(trajectory_df["center_layer"], trajectory_df["trajectory_curvature"], marker="o", color="#4C956C")
        ax1.set_xlabel("center layer")
        ax1.set_ylabel("normalized curvature")
        ax2 = ax1.twinx()
        ax2.plot(trajectory_df["center_layer"], trajectory_df["turn_angle_degrees"], marker="s", color="#B56576")
        ax2.set_ylabel("turn angle (degrees)")
        ax1.set_title("Trajectory curvature of relation geometry across layers")
        plt.tight_layout()
        plt.savefig(diag_dir / "relation_trajectory_curvature.png", dpi=180)
        plt.show()

    return frob_df, frob_rms_df, layer_score_df, trajectory_df


def augment_texts(texts, seed, drop_prob=AUGMENT_DROP_PROB):
    rng = np.random.default_rng(seed)
    augmented = []
    for text in texts:
        words = str(text).split()
        if len(words) <= 1:
            augmented.append(str(text))
            continue
        kept = [word for word in words if rng.random() > drop_prob]
        if not kept:
            kept = [words[int(rng.integers(0, len(words)))]]
        augmented.append(" ".join(kept))
    return augmented


def pairwise_relation_vector_for_texts(
    model, tokenizer, texts, batch_size, max_length, device, training_mode=False, seed=None, desc=None
):
    if seed is not None:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    embeddings = collect_embeddings(
        model, tokenizer, texts, batch_size, max_length, device,
        pooling="cls", training_mode=training_mode, desc=desc
    )
    return relation_vector_from_embeddings(embeddings).numpy()


def paired_relation_values_for_texts(
    model, tokenizer, left_texts, right_texts, batch_size, max_length, device,
    pooling="cls", training_mode=False, seed=None, desc_prefix="paired relation"
):
    if seed is not None:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    left = collect_embeddings(
        model, tokenizer, left_texts, batch_size, max_length, device, pooling=pooling,
        training_mode=training_mode, desc=f"{desc_prefix}: left"
    )
    right = collect_embeddings(
        model, tokenizer, right_texts, batch_size, max_length, device, pooling=pooling,
        training_mode=training_mode, desc=f"{desc_prefix}: right"
    )
    return F.cosine_similarity(left, right).numpy()


def teacher_reference_relation_from_cache(cache_path, sample_indices):
    cache_path = Path(cache_path)
    if not cache_path.exists():
        return None, f"missing teacher cache: {cache_path}"
    try:
        cached = torch.load(cache_path, map_location="cpu")
    except Exception as exc:
        return None, f"could not load teacher cache {cache_path}: {exc}"
    if not torch.is_tensor(cached):
        return None, f"teacher cache is not a tensor: {type(cached)}"
    max_idx = max(sample_indices) if sample_indices else -1
    if len(cached) <= max_idx:
        return None, f"teacher cache has {len(cached)} rows but sample needs index {max_idx}"
    teacher_emb = F.normalize(cached[sample_indices].float(), dim=-1)
    return relation_vector_from_embeddings(teacher_emb).numpy(), str(cache_path)


def relation_stability_analysis(
    model, tokenizer, texts, domains, sample_indices, batch_size, max_length, device, cache_path, diag_dir
):
    if len(texts) < 3:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), {"status": "skipped: fewer than 3 texts"}

    clean_relation = pairwise_relation_vector_for_texts(
        model, tokenizer, texts, batch_size, max_length, device,
        training_mode=False, seed=RANDOM_SEED, desc="Clean relation vector"
    )

    dropout_relations = []
    augmentation_relations = []
    for repeat_idx in range(RELATION_REPEATS):
        seed = RANDOM_SEED + 1000 + repeat_idx
        dropout_relations.append(pairwise_relation_vector_for_texts(
            model, tokenizer, texts, batch_size, max_length, device,
            training_mode=True, seed=seed, desc=f"Dropout relation replicate {repeat_idx + 1}/{RELATION_REPEATS}"
        ))
        aug_texts = augment_texts(texts, seed=RANDOM_SEED + 2000 + repeat_idx)
        augmentation_relations.append(pairwise_relation_vector_for_texts(
            model, tokenizer, aug_texts, batch_size, max_length, device,
            training_mode=False, seed=seed, desc=f"Augmented relation replicate {repeat_idx + 1}/{RELATION_REPEATS}"
        ))

    dropout_relations = np.stack(dropout_relations, axis=0)
    augmentation_relations = np.stack(augmentation_relations, axis=0)
    dropout_var = dropout_relations.var(axis=0)
    augmentation_var = augmentation_relations.var(axis=0)
    combined_var = 0.5 * (dropout_var + augmentation_var)
    mean_perturbed_relation = 0.5 * (dropout_relations.mean(axis=0) + augmentation_relations.mean(axis=0))

    reference_relation, reference_source = teacher_reference_relation_from_cache(cache_path, sample_indices)
    if reference_relation is None:
        reference_relation = clean_relation
        reference_source = f"clean_student_relation fallback ({reference_source})"

    low_thr = float(np.quantile(combined_var, STABLE_RELATION_QUANTILE))
    high_thr = float(np.quantile(combined_var, UNSTABLE_RELATION_QUANTILE))
    stable_group = np.where(combined_var <= low_thr, "stable", np.where(combined_var >= high_thr, "unstable", "middle"))
    error = np.abs(mean_perturbed_relation - reference_relation)

    n = len(texts)
    pair_i, pair_j = torch.triu_indices(n, n, offset=1).numpy()
    domain_i = np.array(domains, dtype=object)[pair_i]
    domain_j = np.array(domains, dtype=object)[pair_j]
    domain_pair = ["::".join(sorted((str(a), str(b)))) for a, b in zip(domain_i, domain_j)]

    pair_df = pd.DataFrame({
        "i": pair_i,
        "j": pair_j,
        "domain_i": domain_i,
        "domain_j": domain_j,
        "domain_pair": domain_pair,
        "same_domain": domain_i == domain_j,
        "clean_relation": clean_relation,
        "mean_perturbed_relation": mean_perturbed_relation,
        "dropout_variance": dropout_var,
        "augmentation_variance": augmentation_var,
        "combined_variance": combined_var,
        "reference_relation": reference_relation,
        "preservation_abs_error": error,
        "stability_group": stable_group,
        "is_stable": stable_group == "stable",
        "is_unstable": stable_group == "unstable",
    })
    pair_df.to_csv(diag_dir / "relation_stability_pairs.csv", index=False)

    group_df = pair_df.groupby("stability_group").agg(
        n_pairs=("combined_variance", "size"),
        mean_combined_variance=("combined_variance", "mean"),
        median_combined_variance=("combined_variance", "median"),
        mean_preservation_abs_error=("preservation_abs_error", "mean"),
        median_preservation_abs_error=("preservation_abs_error", "median"),
        mean_dropout_variance=("dropout_variance", "mean"),
        mean_augmentation_variance=("augmentation_variance", "mean"),
        same_domain_fraction=("same_domain", "mean"),
    ).reset_index()
    group_df["reference_source"] = reference_source
    group_df.to_csv(diag_dir / "stable_vs_unstable_relation_error.csv", index=False)

    domain_df = pair_df.groupby("domain_pair").agg(
        n_pairs=("combined_variance", "size"),
        stable_fraction=("is_stable", "mean"),
        unstable_fraction=("is_unstable", "mean"),
        mean_combined_variance=("combined_variance", "mean"),
        median_combined_variance=("combined_variance", "median"),
        mean_preservation_abs_error=("preservation_abs_error", "mean"),
        median_preservation_abs_error=("preservation_abs_error", "median"),
    ).reset_index().sort_values(["stable_fraction", "n_pairs"], ascending=[False, False])
    domain_df.to_csv(diag_dir / "domain_stable_relation_train_pairs.csv", index=False)

    ordered_group = [g for g in ["stable", "middle", "unstable"] if g in group_df["stability_group"].tolist()]
    if ordered_group:
        plot_df = group_df.set_index("stability_group").loc[ordered_group]
        ax = plot_df[["mean_dropout_variance", "mean_augmentation_variance"]].plot(
            kind="bar", figsize=(8, 4), color=["#4C78A8", "#F58518"]
        )
        ax.set_ylabel("mean relation variance")
        ax.set_xlabel("relation stability group")
        ax.set_title("Relation variance under dropout and augmentation")
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.savefig(diag_dir / "relation_variance_stability_groups.png", dpi=180)
        plt.show()

        ax = plot_df["mean_preservation_abs_error"].plot(kind="bar", figsize=(7, 4), color="#D1495B")
        ax.set_ylabel("mean absolute preservation error")
        ax.set_xlabel("relation stability group")
        ax.set_title("Stable-pair vs unstable-pair preservation error")
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.savefig(diag_dir / "stable_vs_unstable_relation_error.png", dpi=180)
        plt.show()

    if not domain_df.empty:
        top_domain_df = domain_df[domain_df["n_pairs"] >= max(10, int(0.005 * len(pair_df)))].head(12)
        if not top_domain_df.empty:
            ax = top_domain_df.set_index("domain_pair")["stable_fraction"].plot(kind="barh", figsize=(9, 5), color="#4C956C")
            ax.set_xlabel("stable relation fraction")
            ax.set_ylabel("domain pair")
            ax.set_title("Domain-stable relation pairs in training sample")
            plt.tight_layout()
            plt.savefig(diag_dir / "domain_stable_relation_train_pairs.png", dpi=180)
            plt.show()

    summary = {
        "status": "ok",
        "n_texts": len(texts),
        "n_pairs": int(len(pair_df)),
        "relation_repeats": RELATION_REPEATS,
        "stable_quantile": STABLE_RELATION_QUANTILE,
        "unstable_quantile": UNSTABLE_RELATION_QUANTILE,
        "stable_threshold_combined_variance": low_thr,
        "unstable_threshold_combined_variance": high_thr,
        "reference_source": reference_source,
    }
    return pair_df, group_df, domain_df, summary


def sts_domain_stability_analysis(
    model, tokenizer, sts_paths, batch_size, max_length, device, diag_dir,
    teacher_model=None, teacher_tokenizer=None, teacher_device=None, teacher_pooling="last_token"
):
    rows = []
    for sts_path in sts_paths:
        path = Path(sts_path)
        if not path.exists():
            continue
        df = pd.read_csv(path).dropna(subset=["sentence1", "sentence2", "score"])
        if STS_STABILITY_SAMPLE_PER_FILE and len(df) > STS_STABILITY_SAMPLE_PER_FILE:
            df = df.sample(n=STS_STABILITY_SAMPLE_PER_FILE, random_state=RANDOM_SEED)

        left = df["sentence1"].astype(str).tolist()
        right = df["sentence2"].astype(str).tolist()
        gold_relation = (df["score"].astype(float).to_numpy() / 2.5) - 1.0

        clean_relation = paired_relation_values_for_texts(
            model, tokenizer, left, right, batch_size, max_length, device,
            training_mode=False, seed=RANDOM_SEED, desc_prefix=f"{path.stem} clean"
        )

        dropout_relations = []
        augmentation_relations = []
        for repeat_idx in range(STS_STABILITY_REPEATS):
            seed = RANDOM_SEED + 3000 + repeat_idx
            dropout_relations.append(paired_relation_values_for_texts(
                model, tokenizer, left, right, batch_size, max_length, device,
                training_mode=True, seed=seed, desc_prefix=f"{path.stem} dropout {repeat_idx + 1}/{STS_STABILITY_REPEATS}"
            ))
            aug_left = augment_texts(left, seed=RANDOM_SEED + 4000 + repeat_idx)
            aug_right = augment_texts(right, seed=RANDOM_SEED + 5000 + repeat_idx)
            augmentation_relations.append(paired_relation_values_for_texts(
                model, tokenizer, aug_left, aug_right, batch_size, max_length, device,
                training_mode=False, seed=seed, desc_prefix=f"{path.stem} augmentation {repeat_idx + 1}/{STS_STABILITY_REPEATS}"
            ))

        dropout_relations = np.stack(dropout_relations, axis=0)
        augmentation_relations = np.stack(augmentation_relations, axis=0)
        combined_var = 0.5 * (dropout_relations.var(axis=0) + augmentation_relations.var(axis=0))
        stable_thr = float(np.quantile(combined_var, STABLE_RELATION_QUANTILE))
        unstable_thr = float(np.quantile(combined_var, UNSTABLE_RELATION_QUANTILE))
        stable_mask = combined_var <= stable_thr
        unstable_mask = combined_var >= unstable_thr

        reference_relation = gold_relation
        reference_source = "gold_score"
        if teacher_model is not None and teacher_tokenizer is not None and teacher_device is not None:
            try:
                t_left = collect_embeddings(
                    teacher_model, teacher_tokenizer, left, batch_size, max_length, teacher_device,
                    pooling=teacher_pooling, training_mode=False, desc=f"{path.stem} teacher left"
                )
                t_right = collect_embeddings(
                    teacher_model, teacher_tokenizer, right, batch_size, max_length, teacher_device,
                    pooling=teacher_pooling, training_mode=False, desc=f"{path.stem} teacher right"
                )
                reference_relation = F.cosine_similarity(t_left, t_right).numpy()
                reference_source = "teacher_relation"
            except Exception as exc:
                print(f"Teacher relation reference failed for {sts_path}; using gold score instead: {exc}")

        corr_all = safe_corr(clean_relation, gold_relation)
        stable_preservation = safe_corr(clean_relation[stable_mask], reference_relation[stable_mask])
        unstable_preservation = safe_corr(clean_relation[unstable_mask], reference_relation[unstable_mask])
        stable_error = float(np.mean(np.abs(clean_relation[stable_mask] - reference_relation[stable_mask]))) if stable_mask.any() else np.nan
        unstable_error = float(np.mean(np.abs(clean_relation[unstable_mask] - reference_relation[unstable_mask]))) if unstable_mask.any() else np.nan

        rows.append({
            "dataset": sts_path,
            "n_pairs": int(len(df)),
            "reference_source": reference_source,
            "ood_score_student_vs_gold_spearman": corr_all["spearman"],
            "median_relation_variance": float(np.median(combined_var)),
            "stable_relation_fraction": float(stable_mask.mean()),
            "unstable_relation_fraction": float(unstable_mask.mean()),
            "stable_relation_preservation_spearman": stable_preservation["spearman"],
            "unstable_relation_preservation_spearman": unstable_preservation["spearman"],
            "stable_relation_error": stable_error,
            "unstable_relation_error": unstable_error,
            "stable_variance_threshold": stable_thr,
            "unstable_variance_threshold": unstable_thr,
        })

    domain_stability_df = pd.DataFrame(rows)
    domain_stability_df.to_csv(diag_dir / "domain_stable_relation_sts.csv", index=False)

    corr_summary = safe_corr(
        domain_stability_df.get("stable_relation_preservation_spearman", pd.Series(dtype=float)),
        domain_stability_df.get("ood_score_student_vs_gold_spearman", pd.Series(dtype=float)),
    ) if not domain_stability_df.empty else {"pearson": np.nan, "spearman": np.nan, "n": 0}

    with open(diag_dir / "stable_preservation_vs_ood_correlation.json", "w") as f:
        json.dump(corr_summary, f, indent=2)

    if not domain_stability_df.empty:
        ax = domain_stability_df.set_index("dataset")[["stable_relation_error", "unstable_relation_error"]].plot(
            kind="bar", figsize=(12, 4), color=["#4C956C", "#D1495B"]
        )
        ax.set_ylabel("absolute relation error")
        ax.set_title("Stable vs unstable relation error across OOD STS domains")
        plt.xticks(rotation=35, ha="right")
        plt.tight_layout()
        plt.savefig(diag_dir / "domain_stable_relation_sts_error.png", dpi=180)
        plt.show()

        plt.figure(figsize=(6, 5))
        plt.scatter(
            domain_stability_df["stable_relation_preservation_spearman"],
            domain_stability_df["ood_score_student_vs_gold_spearman"],
            s=60, color="#2A6F97", alpha=0.8,
        )
        for _, row in domain_stability_df.iterrows():
            plt.annotate(Path(row["dataset"]).stem, (row["stable_relation_preservation_spearman"], row["ood_score_student_vs_gold_spearman"]), fontsize=8)
        plt.xlabel("stable relation preservation Spearman")
        plt.ylabel("OOD STS Spearman")
        plt.title("Stable relation preservation vs OOD score")
        plt.tight_layout()
        plt.savefig(diag_dir / "stable_preservation_vs_ood_score.png", dpi=180)
        plt.show()

    return domain_stability_df, corr_summary


def load_teacher_for_sts(model_name, dtype_name):
    teacher_device = torch.device("cuda:1" if torch.cuda.device_count() > 1 else device)
    kwargs = {"trust_remote_code": True}
    if teacher_device.type == "cuda":
        if dtype_name == "bfloat16":
            kwargs["torch_dtype"] = torch.bfloat16
        elif dtype_name == "float16":
            kwargs["torch_dtype"] = torch.float16
    teacher_tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    teacher_model = AutoModel.from_pretrained(model_name, **kwargs)
    teacher_model.to(teacher_device)
    teacher_model.eval()
    for p in teacher_model.parameters():
        p.requires_grad_(False)
    return teacher_model, teacher_tokenizer, teacher_device


# 1) Layer CKA: are student layers becoming too similar?
train_df = pd.read_csv(train_data_path)
all_train_texts = get_first_side_texts(train_df, task_type)
if "source" in train_df.columns:
    all_train_domains = train_df["source"].fillna("unknown").astype(str).tolist()
else:
    all_train_domains = ["train"] * len(all_train_texts)

rng = np.random.default_rng(RANDOM_SEED)
if len(all_train_texts) > TRAIN_SAMPLE_TEXTS:
    train_sample_indices = rng.choice(len(all_train_texts), size=TRAIN_SAMPLE_TEXTS, replace=False).tolist()
else:
    train_sample_indices = list(range(len(all_train_texts)))
train_texts = [all_train_texts[i] for i in train_sample_indices]
train_domains = [all_train_domains[i] for i in train_sample_indices]

cls_layers = collect_student_cls_layers(
    student_model, student_tokenizer, train_texts, DIAG_BATCH_SIZE, max_length, device
)
num_layers = len(cls_layers)
cka = np.zeros((num_layers, num_layers), dtype=np.float32)
for i in range(num_layers):
    for j in range(i, num_layers):
        value = linear_cka(cls_layers[i], cls_layers[j])
        cka[i, j] = cka[j, i] = value

cka_df = pd.DataFrame(cka, index=[f"L{i}" for i in range(num_layers)], columns=[f"L{i}" for i in range(num_layers)])
cka_df.to_csv(DIAG_DIR / "layer_cka.csv")

plt.figure(figsize=(9, 7))
plt.imshow(cka_df.values, vmin=0, vmax=1, cmap="magma")
plt.colorbar(label="linear CKA")
plt.xticks(range(num_layers), cka_df.columns, rotation=90)
plt.yticks(range(num_layers), cka_df.index)
plt.title("Layer CKA on student CLS states")
plt.tight_layout()
plt.savefig(DIAG_DIR / "layer_cka.png", dpi=180)
plt.show()

# B) Geometry smoothing vs semantic hierarchy.
# R_l is the in-batch relation matrix for layer l. D_ij is reported both as the
# raw Frobenius norm and as RMS-normalized distance so runs with different probe
# sample sizes remain comparable.
relation_distance_df, relation_distance_rms_df, layer_geometry_score_df, trajectory_curvature_df = relation_geometry_diagnostics(
    cls_layers, cka, DIAG_DIR
)

# 2) Pair-group contribution: top/middle/bottom relation pairs in LLASD.
pair_group_df = pair_group_contribution(cls_layers, start_rkd=start_rkd)
pair_group_df.to_csv(DIAG_DIR / "pair_group_contribution.csv", index=False)

if not pair_group_df.empty:
    group_order = ["bottom", "middle", "top"]
    pivot = pair_group_df.pivot_table(index="layer_pair", columns="group", values="frac_sse", aggfunc="sum").fillna(0)
    pivot = pivot[[g for g in group_order if g in pivot.columns]]
    ax = pivot.plot(kind="bar", stacked=True, figsize=(12, 4), color=["#53777A", "#ECD078", "#C02942"][:len(pivot.columns)])
    ax.set_ylabel("fraction of LLASD squared error")
    ax.set_xlabel("consecutive student layer pair")
    ax.set_title("LLASD pair-group contribution by target similarity quantile")
    ax.legend(title="target relation group")
    plt.tight_layout()
    plt.savefig(DIAG_DIR / "pair_group_contribution.png", dpi=180)
    plt.show()

# 3) Gradient norm by layer: does LLASD press shallow layers strongly?
grad_df = llasd_grad_norm_by_layer(
    student_model, student_tokenizer, train_texts, GRAD_BATCH_SIZE, max_length, device, start_rkd=start_rkd
)
grad_df.to_csv(DIAG_DIR / "grad_norm_by_layer.csv", index=False)

plot_grad_df = grad_df[grad_df["layer"] != "other"].copy()
if not plot_grad_df.empty:
    plt.figure(figsize=(10, 4))
    plt.bar(plot_grad_df["layer"], plot_grad_df["grad_norm"], color="#345995")
    plt.yscale("log")
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("LLASD-only grad norm (log scale)")
    plt.title("Gradient norm by student layer from loss_struct only")
    plt.tight_layout()
    plt.savefig(DIAG_DIR / "grad_norm_by_layer.png", dpi=180)
    plt.show()

# 4) Relation calibration on STS: student vs teacher and gold/final relation.
teacher_model = teacher_tokenizer = teacher_device = None
if RUN_TEACHER_STS_CALIBRATION:
    try:
        teacher_model, teacher_tokenizer, teacher_device = load_teacher_for_sts(teacher_model_name, teacher_dtype_name)
    except Exception as exc:
        print(f"Teacher STS calibration disabled because teacher load failed: {exc}")
        RUN_TEACHER_STS_CALIBRATION = False

calibration_rows = []
scatter_frames = []
sts_paths = list(eval_sts_tasks) + list(test_sts_tasks)
for sts_path in sts_paths:
    path = Path(sts_path)
    if not path.exists():
        print(f"Skipping missing STS file: {sts_path}")
        continue
    df = pd.read_csv(path).dropna(subset=["sentence1", "sentence2", "score"])
    if STS_SAMPLE_PER_FILE and len(df) > STS_SAMPLE_PER_FILE:
        df = df.sample(n=STS_SAMPLE_PER_FILE, random_state=RANDOM_SEED)

    s1 = df["sentence1"].astype(str).tolist()
    s2 = df["sentence2"].astype(str).tolist()
    gold_relation = (df["score"].astype(float).to_numpy() / 2.5) - 1.0

    emb1 = collect_embeddings(student_model, student_tokenizer, s1, DIAG_BATCH_SIZE, max_length, device, pooling="cls")
    emb2 = collect_embeddings(student_model, student_tokenizer, s2, DIAG_BATCH_SIZE, max_length, device, pooling="cls")
    student_relation = F.cosine_similarity(emb1, emb2).numpy()

    row = {"dataset": sts_path, "n": len(df)}
    sg = safe_corr(student_relation, gold_relation)
    row.update({"student_vs_gold_pearson": sg["pearson"], "student_vs_gold_spearman": sg["spearman"]})

    frame = pd.DataFrame({
        "dataset": sts_path,
        "student_relation": student_relation,
        "gold_relation": gold_relation,
    })

    if RUN_TEACHER_STS_CALIBRATION and teacher_model is not None:
        t_emb1 = collect_embeddings(teacher_model, teacher_tokenizer, s1, DIAG_BATCH_SIZE, max_length, teacher_device, pooling=teacher_pooling)
        t_emb2 = collect_embeddings(teacher_model, teacher_tokenizer, s2, DIAG_BATCH_SIZE, max_length, teacher_device, pooling=teacher_pooling)
        teacher_relation = F.cosine_similarity(t_emb1, t_emb2).numpy()
        st = safe_corr(student_relation, teacher_relation)
        tg = safe_corr(teacher_relation, gold_relation)
        row.update({
            "student_vs_teacher_pearson": st["pearson"],
            "student_vs_teacher_spearman": st["spearman"],
            "teacher_vs_gold_pearson": tg["pearson"],
            "teacher_vs_gold_spearman": tg["spearman"],
            "student_teacher_mae": float(np.mean(np.abs(student_relation - teacher_relation))),
        })
        frame["teacher_relation"] = teacher_relation

    calibration_rows.append(row)
    scatter_frames.append(frame)

calibration_df = pd.DataFrame(calibration_rows)
calibration_df.to_csv(DIAG_DIR / "relation_calibration_sts.csv", index=False)
if scatter_frames:
    scatter_df = pd.concat(scatter_frames, ignore_index=True)
    scatter_df.to_csv(DIAG_DIR / "relation_calibration_sts_points.csv", index=False)
else:
    scatter_df = pd.DataFrame()

if not calibration_df.empty:
    metric_cols = [
        c for c in [
            "student_vs_gold_spearman",
            "student_vs_teacher_spearman",
            "teacher_vs_gold_spearman",
        ] if c in calibration_df.columns
    ]
    if metric_cols:
        ax = calibration_df.set_index("dataset")[metric_cols].plot(kind="bar", figsize=(12, 4))
        ax.set_ylim(-1, 1)
        ax.set_ylabel("Spearman correlation")
        ax.set_title("STS relation calibration")
        plt.xticks(rotation=35, ha="right")
        plt.tight_layout()
        plt.savefig(DIAG_DIR / "relation_calibration_sts.png", dpi=180)
        plt.show()

if not scatter_df.empty:
    plt.figure(figsize=(6, 5))
    plt.scatter(scatter_df["gold_relation"], scatter_df["student_relation"], s=10, alpha=0.35, label="student vs gold")
    if "teacher_relation" in scatter_df.columns:
        plt.scatter(scatter_df["teacher_relation"], scatter_df["student_relation"], s=10, alpha=0.25, label="student vs teacher")
    plt.xlabel("reference relation")
    plt.ylabel("student relation")
    plt.title("Relation calibration scatter")
    plt.legend()
    plt.tight_layout()
    plt.savefig(DIAG_DIR / "relation_calibration_scatter.png", dpi=180)
    plt.show()

# C) Stable vs unstable relation analysis.
stability_pair_df = pd.DataFrame()
stability_group_df = pd.DataFrame()
domain_stability_train_df = pd.DataFrame()
stability_summary = {"status": "disabled"}
if RUN_STABILITY_DIAGNOSTICS:
    stability_n = min(RELATION_STABILITY_SAMPLE_TEXTS, len(train_texts))
    stability_texts = train_texts[:stability_n]
    stability_domains = train_domains[:stability_n]
    stability_indices = train_sample_indices[:stability_n]
    stability_pair_df, stability_group_df, domain_stability_train_df, stability_summary = relation_stability_analysis(
        student_model, student_tokenizer, stability_texts, stability_domains, stability_indices,
        DIAG_BATCH_SIZE, max_length, device, cfg.get("cache_path", "cache/teacher_train.pt"), DIAG_DIR
    )

domain_stability_sts_df = pd.DataFrame()
stable_preservation_ood_corr = {"pearson": np.nan, "spearman": np.nan, "n": 0}
if RUN_DOMAIN_STABILITY_ON_STS:
    domain_stability_sts_df, stable_preservation_ood_corr = sts_domain_stability_analysis(
        student_model, student_tokenizer, sts_paths, DIAG_BATCH_SIZE, max_length, device, DIAG_DIR,
        teacher_model=teacher_model if RUN_TEACHER_STS_CALIBRATION else None,
        teacher_tokenizer=teacher_tokenizer if RUN_TEACHER_STS_CALIBRATION else None,
        teacher_device=teacher_device if RUN_TEACHER_STS_CALIBRATION else None,
        teacher_pooling=teacher_pooling,
    )

summary = {
    "checkpoint_path": str(checkpoint_path),
    "student_model_name": student_model_name,
    "teacher_model_name": teacher_model_name,
    "train_sample_texts": len(train_texts),
    "sts_sample_per_file": STS_SAMPLE_PER_FILE,
    "num_student_hidden_layers": num_layers,
    "llasd_definition": "loss_struct = mean MSE between in-batch relation matrices of consecutive student CLS layers",
    "artifacts": {
        "training_log": str(DIAG_DIR / "train_talas.log"),
        "layer_cka_csv": str(DIAG_DIR / "layer_cka.csv"),
        "layer_cka_png": str(DIAG_DIR / "layer_cka.png"),
        "relation_geometry_distance_csv": str(DIAG_DIR / "relation_geometry_distance_frobenius.csv"),
        "relation_geometry_distance_rms_csv": str(DIAG_DIR / "relation_geometry_distance_rms.csv"),
        "relation_geometry_distance_png": str(DIAG_DIR / "relation_geometry_distance_heatmap.png"),
        "layer_geometry_scores_csv": str(DIAG_DIR / "layer_geometry_scores.csv"),
        "layer_geometry_scores_png": str(DIAG_DIR / "layer_geometry_scores.png"),
        "trajectory_curvature_csv": str(DIAG_DIR / "relation_trajectory_curvature.csv"),
        "trajectory_curvature_png": str(DIAG_DIR / "relation_trajectory_curvature.png"),
        "pair_group_csv": str(DIAG_DIR / "pair_group_contribution.csv"),
        "pair_group_png": str(DIAG_DIR / "pair_group_contribution.png"),
        "grad_norm_csv": str(DIAG_DIR / "grad_norm_by_layer.csv"),
        "grad_norm_png": str(DIAG_DIR / "grad_norm_by_layer.png"),
        "relation_calibration_csv": str(DIAG_DIR / "relation_calibration_sts.csv"),
        "relation_calibration_points_csv": str(DIAG_DIR / "relation_calibration_sts_points.csv"),
        "relation_calibration_png": str(DIAG_DIR / "relation_calibration_sts.png"),
        "relation_stability_pairs_csv": str(DIAG_DIR / "relation_stability_pairs.csv"),
        "stable_vs_unstable_error_csv": str(DIAG_DIR / "stable_vs_unstable_relation_error.csv"),
        "stable_vs_unstable_error_png": str(DIAG_DIR / "stable_vs_unstable_relation_error.png"),
        "relation_variance_groups_png": str(DIAG_DIR / "relation_variance_stability_groups.png"),
        "domain_stable_train_csv": str(DIAG_DIR / "domain_stable_relation_train_pairs.csv"),
        "domain_stable_train_png": str(DIAG_DIR / "domain_stable_relation_train_pairs.png"),
        "domain_stable_sts_csv": str(DIAG_DIR / "domain_stable_relation_sts.csv"),
        "domain_stable_sts_error_png": str(DIAG_DIR / "domain_stable_relation_sts_error.png"),
        "stable_preservation_ood_corr_json": str(DIAG_DIR / "stable_preservation_vs_ood_correlation.json"),
        "stable_preservation_ood_png": str(DIAG_DIR / "stable_preservation_vs_ood_score.png"),
    },
    "stability_summary": stability_summary,
    "stable_preservation_ood_correlation": stable_preservation_ood_corr,
}
with open(DIAG_DIR / "diagnostics_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

if teacher_model is not None:
    del teacher_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nSaved TALAS diagnostics to:", DIAG_DIR.resolve())
for path in summary["artifacts"].values():
    print("-", path)

print("\nLayer CKA (last 5x5):")
try:
    display(cka_df.iloc[-5:, -5:])
except NameError:
    print(cka_df.iloc[-5:, -5:].to_string())

print("\nRelation geometry distance RMS (last 5x5):")
try:
    display(relation_distance_rms_df.iloc[-5:, -5:])
except NameError:
    print(relation_distance_rms_df.iloc[-5:, -5:].to_string())

print("\nLayer geometry scores:")
try:
    display(layer_geometry_score_df)
except NameError:
    print(layer_geometry_score_df.to_string(index=False))

print("\nTrajectory curvature:")
try:
    display(trajectory_curvature_df)
except NameError:
    print(trajectory_curvature_df.to_string(index=False))

print("\nPair-group contribution preview:")
try:
    display(pair_group_df.head(12))
except NameError:
    print(pair_group_df.head(12).to_string(index=False))

print("\nGradient norm by layer:")
try:
    display(grad_df)
except NameError:
    print(grad_df.to_string(index=False))

print("\nSTS relation calibration:")
try:
    display(calibration_df)
except NameError:
    print(calibration_df.to_string(index=False))


print("\nStable vs unstable relation groups:")
try:
    display(stability_group_df)
except NameError:
    print(stability_group_df.to_string(index=False) if not stability_group_df.empty else "No stability groups computed.")

print("\nDomain-stable train relation preview:")
try:
    display(domain_stability_train_df.head(12))
except NameError:
    print(domain_stability_train_df.head(12).to_string(index=False) if not domain_stability_train_df.empty else "No train domain stability computed.")

print("\nDomain-stable STS relation analysis:")
try:
    display(domain_stability_sts_df)
except NameError:
    print(domain_stability_sts_df.to_string(index=False) if not domain_stability_sts_df.empty else "No STS domain stability computed.")

print("\nCorrelation between stable relation preservation and OOD score:")
print(stable_preservation_ood_corr)


In [ ]:
# Run all validation and test benchmarks from the saved TALAS checkpoint
import re
from pathlib import Path

import torch
from transformers import AutoModel

from src.evaluation.evaluation_automodel import (
    eval_classification_task,
    eval_pair_task,
    eval_sts_task,
    eval_cls_tasks,
    eval_pair_tasks,
    eval_sts_tasks,
    test_cls_tasks,
    test_pair_tasks,
    test_sts_tasks,
)

checkpoint_dir = Path("checkpoints/talas")
best_checkpoint = checkpoint_dir / "best_model.pt"

if best_checkpoint.exists():
    checkpoint_path = best_checkpoint
else:
    checkpoints = sorted(
        checkpoint_dir.glob("checkpoint_epoch_*.pt"),
        key=lambda p: int(re.search(r"checkpoint_epoch_(\d+)", p.stem).group(1)),
    )
    assert checkpoints, f"No checkpoint found in {checkpoint_dir}"
    checkpoint_path = checkpoints[-1]

print(f"Loading checkpoint: {checkpoint_path}")

try:
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
except TypeError:
    checkpoint = torch.load(checkpoint_path, map_location="cpu")

cfg = checkpoint.get("config", {})
student_model_name = cfg.get("student_model_name", "google-bert/bert-base-uncased")
print(f"Student model: {student_model_name}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = AutoModel.from_pretrained(student_model_name)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()

benchmark_groups = [
    ("Validation classification", eval_classification_task, eval_cls_tasks),
    ("Validation pair classification", eval_pair_task, eval_pair_tasks),
    ("Validation STS", eval_sts_task, eval_sts_tasks),
    ("Test classification", eval_classification_task, test_cls_tasks),
    ("Test pair classification", eval_pair_task, test_pair_tasks),
    ("Test STS", eval_sts_task, test_sts_tasks),
]

for name, eval_fn, tasks in benchmark_groups:
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)
    eval_fn(model, tasks)

print("\nAll benchmark evaluations finished.")

Loading checkpoint: checkpoints/talas/best_model.pt
Student model: google-bert/bert-base-uncased
Using device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Validation classification
 eval classifier
data/multi-data/banking77_validation.csv


100%|██████████| 16/16 [00:00<00:00, 42.18it/s]
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


...

Spearman: 0.7849
data/multi-data/sts12_test.csv


100%|██████████| 49/49 [00:01<00:00, 24.85it/s]


Spearman: 0.7278
data/multi-data/stsb_test.csv


100%|██████████| 22/22 [00:00<00:00, 24.31it/s]

Spearman: 0.7997

All benchmark evaluations finished.
